# Milestone 1 Exploration

The goal of this notebook is to explore a small subset of the Amazon Movies and TV review and metadata files, understand the available fields, and justify the field selection and preprocessing decisions for retrieval.

In [3]:
import gzip
import json
import pandas as pd

review_path = "../data/raw/Movies_and_TV.jsonl.gz"
meta_path = "../data/raw/meta_Movies_and_TV.jsonl.gz"

meta_lookup = {}

with gzip.open(meta_path, "rt", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        parent_asin = str(record.get("parent_asin", "")).strip()

        raw_title = record.get("title", "")
        title = ""
        if raw_title is not None:
            title = str(raw_title).strip()

        if parent_asin and title:
            meta_lookup[parent_asin] = record

print("Metadata products with titles:", len(meta_lookup))

Metadata products with titles: 434222


In [4]:
matched_reviews = []
matched_asins = set()

with gzip.open(review_path, "rt", encoding="utf-8") as f:
    for line in f:
        review = json.loads(line)
        parent_asin = str(review.get("parent_asin", "")).strip()

        if parent_asin in meta_lookup:
            matched_reviews.append(review)
            matched_asins.add(parent_asin)

            if len(matched_reviews) == 200:
                break

reviews_sample = pd.DataFrame(matched_reviews)

In [5]:
matched_meta = [meta_lookup[asin] for asin in matched_asins]
meta_sample = pd.DataFrame(matched_meta)

print("Reviews sample shape:", reviews_sample.shape)
print("Metadata sample shape:", meta_sample.shape)

reviews_sample.head()
meta_sample.head()

Reviews sample shape: (200, 10)
Metadata sample shape: (199, 15)


,main_category,title,subtitle,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Prime Video,"Monsters, Inc.",NaN,4.8,24524,"[IMDb 8.1, 1 h 32 min, 2001, X-Ray, HDR, UHD, G]",[Monsters working at a scream processing facto...,19.99,[{'360w': 'https://images-na.ssl-images-amazon...,[],NaN,"[Animation, Comedy, Heartwarming, Playful]","{'Audio languages': ['English', 'English [Audi...",B00BHU9CCO,None
1,Movies & TV,Die Hard: 4-Film Collection,NaN,4.8,974,[],[NEW DVD SET (CrA204tE233) - Media Title: DIE ...,11.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],"Bruce Willis (Actor), Jeremy Irons (Acto...","[Movies & TV, Genre for Featured Categories, A...","{'Is Discontinued By Manufacturer': 'No', 'MPA...",B00MP2FFL0,None
2,Movies & TV,Blue Hawaii,NaN,4.8,3828,[],"[Blue Hawaii (DVD), The year was 1961. Fallout...",9.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],"Joan Blackman (Actor), Angela Lansbury (...","[Movies & TV, Studio Specials, Warner Home Vid...","{'Genre': 'Musicals & Performing Arts, Drama/L...",B00AEFXK44,None
3,Movies & TV,Chitty Chitty Bang Bang,NaN,4.8,9142,[],"[Chitty Chitty Bang Bang (RPKG/BD), ]]>]",8.77,[{'thumb': 'https://m.media-amazon.com/images/...,[],"Dick Van Dyke (Actor), Sally Ann Howes (...","[Movies & TV, Blu-ray, Movies]","{'Aspect Ratio': '2.20:1', 'Is Discontinued By...",B00EPA3VA2,None
4,Movies & TV,"Home for the Holidays, Cover may vary",NaN,4.7,1160,[],"[Home For the Holidays (RPKG/DVD), ]]>]",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],"Various (Actor, Director) Rated: PG-13 ...","[Movies & TV, Holidays & Seasonal, Christmas, ...","{'Genre': 'Comedy', 'Format': 'Multiple Format...",B00005LOKR,None


## Dataset overview

The first 200 entries are used from both the review file and the metadata file for quick exploration. This is enough to inspect the structure of the data and decide which fields are useful for retrieval, without loading the full dataset into memory.

In [6]:
print("Reviews sample shape:", reviews_sample.shape)
print("Metadata sample shape:", meta_sample.shape)

print("\nReview columns:")
print(reviews_sample.columns.tolist())

print("\nMetadata columns:")
print(meta_sample.columns.tolist())

Reviews sample shape: (200, 10)
Metadata sample shape: (199, 15)

Review columns:
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

Metadata columns:
['main_category', 'title', 'subtitle', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']


In [7]:
reviews_sample.info()
meta_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rating             200 non-null    float64
 1   title              200 non-null    str    
 2   text               200 non-null    str    
 3   images             200 non-null    object 
 4   asin               200 non-null    str    
 5   parent_asin        200 non-null    str    
 6   user_id            200 non-null    str    
 7   timestamp          200 non-null    int64  
 8   helpful_vote       200 non-null    int64  
 9   verified_purchase  200 non-null    bool   
dtypes: bool(1), float64(1), int64(2), object(1), str(5)
memory usage: 90.0+ KB
<class 'pandas.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    196 non-null    str    
 1   title        

## Inspection of Sample Records

In [8]:
reviews_sample[["title", "text", "rating", "parent_asin"]].head(5)

,title,text,rating,parent_asin
0,Five Stars,"Amazon, please buy the show! I'm hooked!",5.0,B013488XFS
1,Five Stars,My Kiddos LOVE this show!!,5.0,B00CB6VTDS
2,Five Stars,Great film all around. Some films about Jesus ...,5.0,B01BZ4DOGQ
3,AWESOME DVD-GREAT FOR BEGINNERS & BUSY PEOPLE,This DVD was GREAT! I am a stay at home mom w...,5.0,B0002J58ME
4,Awesome,Awesome movie! Must see.,5.0,B079FLYB41


In [9]:
meta_sample[["title", "description", "features", "categories", "parent_asin"]].head(5)

,title,description,features,categories,parent_asin
0,"Monsters, Inc.",[Monsters working at a scream processing facto...,"[IMDb 8.1, 1 h 32 min, 2001, X-Ray, HDR, UHD, G]","[Animation, Comedy, Heartwarming, Playful]",B00BHU9CCO
1,Die Hard: 4-Film Collection,[NEW DVD SET (CrA204tE233) - Media Title: DIE ...,[],"[Movies & TV, Genre for Featured Categories, A...",B00MP2FFL0
2,Blue Hawaii,"[Blue Hawaii (DVD), The year was 1961. Fallout...",[],"[Movies & TV, Studio Specials, Warner Home Vid...",B00AEFXK44
3,Chitty Chitty Bang Bang,"[Chitty Chitty Bang Bang (RPKG/BD), ]]>]",[],"[Movies & TV, Blu-ray, Movies]",B00EPA3VA2
4,"Home for the Holidays, Cover may vary","[Home For the Holidays (RPKG/DVD), ]]>]",[],"[Movies & TV, Holidays & Seasonal, Christmas, ...",B00005LOKR


In [10]:
type(meta_sample.loc[0, "description"]), meta_sample.loc[0, "description"]

(list,
 ['Monsters working at a scream processing factory are scared silly by a cute little girl who wanders into Monstropolis. While helping her find her way home, the gang faces monstrous misadventures and the young girl finds her way into their hearts.'])

In [11]:
type(meta_sample.loc[0, "features"]), meta_sample.loc[0, "features"]

(list, ['IMDb 8.1', '1 h 32 min', '2001', 'X-Ray', 'HDR', 'UHD', 'G'])

In [12]:
type(meta_sample.loc[0, "categories"]), meta_sample.loc[0, "categories"]

(list, ['Animation', 'Comedy', 'Heartwarming', 'Playful'])

In [13]:
type(reviews_sample.loc[0, "images"]), reviews_sample.loc[0, "images"]

(list, [])

## Field selection for retrieval

Based on the sample records, different fields are selected from the review and metadata files depending on whether they contain useful natural language text.

For the review data, the most useful fields for retrieval are `title` and `text`. These two fields contain the main user-written content and are the most likely to match search queries effectively. `rating` is kept as metadata because it may still be useful later for filtering or analysis, and `parent_asin` is kept so that reviews can be linked to the corresponding product metadata. Fields such as `user_id` and `timestamp` are not used as main retrieval text because they are identifiers or administrative information rather than descriptive content. The `images` field is also not selected because it is a list and appears empty in the sample records.

For the metadata file, the most useful fields for retrieval are `title`, `description`, `features`, and `categories`. These fields describe what the product is and contain text that could help both keyword-based and semantic retrieval. Since `description`, `features`, and `categories` are stored as lists, they need to be flattened into text during preprocessing. `parent_asin` is also kept as metadata so that metadata records can be joined with reviews. Fields such as `price`, `average_rating`, and `rating_number` may still be useful as metadata, but they are not the main retrieval text. Fields like `subtitle` and `bought_together` are not prioritized because they are sparse in this sample.

## Preprocessing decisions

For preprocessing, only fields that are useful for retrieval or record linkage are kept. In the review data, `title` and `text` are combined into a single retrieval text field so that both the short review summary and the full review body can be searched together. In the metadata data, `title`, `description`, `features`, and `categories` are combined into one text field. Because several metadata fields are stored as lists, they are first converted into plain text by joining their elements.

`parent_asin` is kept as metadata so that review records and metadata records can be connected later. At this stage, preprocessing should stay fairly light. Text can be lowercased, stripped of extra whitespace, and empty retrieval fields can be removed, but overly aggressive cleaning is avoided since wording and punctuation may still be useful for retrieval. Fields such as `user_id`, `timestamp`, and image-related fields are excluded from the main retrieval text because they do not add meaningful semantic content.

In [14]:
reviews_processed = reviews_sample[["parent_asin", "title", "text", "rating"]].copy()

reviews_processed["title"] = reviews_processed["title"].fillna("").str.strip()
reviews_processed["text"] = reviews_processed["text"].fillna("").str.strip()

reviews_processed["retrieval_text"] = (
    reviews_processed["title"] + ". " + reviews_processed["text"]
).str.strip()

reviews_processed = reviews_processed[reviews_processed["retrieval_text"].str.len() > 0]

reviews_processed.head()

,parent_asin,title,text,rating,retrieval_text
0,B013488XFS,Five Stars,"Amazon, please buy the show! I'm hooked!",5.0,"Five Stars. Amazon, please buy the show! I'm h..."
1,B00CB6VTDS,Five Stars,My Kiddos LOVE this show!!,5.0,Five Stars. My Kiddos LOVE this show!!
2,B01BZ4DOGQ,Five Stars,Great film all around. Some films about Jesus ...,5.0,Five Stars. Great film all around. Some films ...
3,B0002J58ME,AWESOME DVD-GREAT FOR BEGINNERS & BUSY PEOPLE,This DVD was GREAT! I am a stay at home mom w...,5.0,AWESOME DVD-GREAT FOR BEGINNERS & BUSY PEOPLE....
4,B079FLYB41,Awesome,Awesome movie! Must see.,5.0,Awesome. Awesome movie! Must see.


In [15]:
meta_processed = meta_sample[["parent_asin", "title", "description", "features", "categories"]].copy()

meta_processed["title"] = meta_processed["title"].fillna("").str.strip()

meta_processed["description"] = meta_processed["description"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)

meta_processed["features"] = meta_processed["features"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)

meta_processed["categories"] = meta_processed["categories"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)

meta_processed["retrieval_text"] = (
    meta_processed["title"] + ". "
    + meta_processed["description"] + " "
    + meta_processed["features"] + " "
    + meta_processed["categories"]
).str.strip()

meta_processed = meta_processed[meta_processed["retrieval_text"].str.len() > 0]

meta_processed.head()

,parent_asin,title,description,features,categories,retrieval_text
0,B00BHU9CCO,"Monsters, Inc.",Monsters working at a scream processing factor...,IMDb 8.1 1 h 32 min 2001 X-Ray HDR UHD G,Animation Comedy Heartwarming Playful,"Monsters, Inc.. Monsters working at a scream p..."
1,B00MP2FFL0,Die Hard: 4-Film Collection,NEW DVD SET (CrA204tE233) - Media Title: DIE H...,,Movies & TV Genre for Featured Categories Acti...,Die Hard: 4-Film Collection. NEW DVD SET (CrA2...
2,B00AEFXK44,Blue Hawaii,Blue Hawaii (DVD) The year was 1961. Fallout s...,,Movies & TV Studio Specials Warner Home Video ...,Blue Hawaii. Blue Hawaii (DVD) The year was 19...
3,B00EPA3VA2,Chitty Chitty Bang Bang,Chitty Chitty Bang Bang (RPKG/BD) ]]>,,Movies & TV Blu-ray Movies,Chitty Chitty Bang Bang. Chitty Chitty Bang Ba...
4,B00005LOKR,"Home for the Holidays, Cover may vary",Home For the Holidays (RPKG/DVD) ]]>,,Movies & TV Holidays & Seasonal Christmas Comedy,"Home for the Holidays, Cover may vary. Home Fo..."


In [16]:
reviews_processed.to_csv("../data/processed/reviews_sample_processed.csv", index=False)
meta_processed.to_csv("../data/processed/meta_sample_processed.csv", index=False)

In [ ]:
import sys
import os
import torch 
import torch.nn as nn
import faiss
import numpy as np

sys.path.append(os.path.abspath('..'))

from src.semantic import load_documents, get_embedding_model, encode_documents, save_semantic_artifacts

print("Combining documents...")
documents = load_documents(
    reviews_path="../data/processed/reviews_sample_processed.csv",
    meta_path="../data/processed/meta_sample_processed.csv"
)

print("Loading model...")
model = get_embedding_model()

print("Encoding documents...")
embeddings = encode_documents(documents, model) 

print("Creating FAISS Index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype('float32'))

print("Saving artifacts to data/processed/...")
save_semantic_artifacts(
    index, 
    documents, 
    index_path="../data/processed/faiss_index.bin", 
    docs_path="../data/processed/combined_documents.csv"
)
print("Success! data/processed/faiss_index.bin has been created.")

/Users/chajingyi/miniforge3/envs/dsci-575-project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Combining documents...
Loading model...


/Users/chajingyi/miniforge3/envs/dsci-575-project/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Encoding documents... this may take a minute.


Batches: 100%|██████████| 7/7 [00:11<00:00,  1.66s/it]


Saving artifacts to data/processed/...


TypeError: Wrong number or type of arguments for overloaded function 'write_index'.
  Possible C/C++ prototypes are:
    faiss::write_index(faiss::Index const *,char const *,int)
    faiss::write_index(faiss::Index const *,char const *)
    faiss::write_index(faiss::Index const *,FILE *,int)
    faiss::write_index(faiss::Index const *,FILE *)
    faiss::write_index(faiss::Index const *,faiss::IOWriter *,int)
    faiss::write_index(faiss::Index const *,faiss::IOWriter *)
